# Re-validate the banked decisions under paired 5-fold CV

Every gain in the current stack was measured on a **single holdout split**:

| step | claimed gain |
|---|---|
| LightGBM -> CatBoost | +0.0059 |
| +5 v2 features (var_cusum_max, kuiper, cvm, skew_shift, kurt_shift) | +0.0040 |
| +variance-GLR (var_glr, var_glr_peak) | +0.0024 |
| **total shipped in submit#3** | **+0.0123** |

Cloud confirmed the *total*: submit#2 0.5884 -> submit#3 0.5984 = +0.010. So the stack as a
whole is real. What a single split cannot tell us is **which links carry it**. Fold-to-fold
spread runs +/-0.008-0.013, which is larger than every individual claim above. A +0.0024
measured once is indistinguishable from a fold that happened to be easy.

That matters because we keep building on these links. If `cvm` or the GLR pair is noise, it is
dead weight in the per-step inference budget and a false lead for "more variance features".

Four configs, **identical folds**, so `paired_compare` can cancel fold difficulty:

- **A** LGB / 43 base feats -> the deployed submit#2
- **B** Cat / 43 -> isolates the learner swap
- **C** Cat / 48 -> isolates the v2 features
- **D** Cat / 50 -> isolates GLR; equals shipped submit#3

Train on the ~40 log-spaced steps (matches deployment), validate on **all** steps of the
held-out series (matches how TS-AUC scores).

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier

TOOLS = os.path.abspath('../tools')
sys.path.insert(0, TOOLS)
import cv_tools as CV

CACHE = os.path.join(TOOLS, 'cv_folds_by_series', 'cv_cache_full.npz')
d = np.load(CACHE)
X, y, sid, tim = d['X'], d['y'], d['sid'], d['time']
sampled, cols = d['is_sampled_step'], list(d['cols'])
index = pd.MultiIndex.from_arrays([sid, tim], names=['id', 'time'])
print(f'{X.shape[0]:,} rows | {len(np.unique(sid)):,} series | {len(cols)} feats | '
      f'sampled {sampled.sum():,} | pos-rate {y.mean():.3f}')

5,036,517 rows | 10,000 series | 50 feats | sampled 328,996 | pos-rate 0.256


In [2]:
# Feature groups. The cache stores all 50 in StreamState order; the older configs are
# strict PREFIXES of it, so an ablation is a column slice - no rebuild needed.
EXTRA = ['var_cusum_max', 'kuiper', 'cvm', 'skew_shift', 'kurt_shift']
GLR = ['var_glr', 'var_glr_peak']
n_v1 = len(cols) - len(EXTRA) - len(GLR)
assert cols[n_v1:n_v1 + len(EXTRA)] == EXTRA and cols[-len(GLR):] == GLR, 'column order moved'
V1, V2, V3 = slice(0, n_v1), slice(0, n_v1 + len(EXTRA)), slice(0, len(cols))
print(f'v1 {n_v1} feats | v2 {n_v1 + len(EXTRA)} | v3 {len(cols)}')

LGBP = dict(objective='binary', n_estimators=1500, learning_rate=0.02, num_leaves=63,
            min_child_samples=300, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            reg_lambda=1.0, n_jobs=-1, verbosity=-1, random_state=0)
CATP = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
            bootstrap_type='Bernoulli', subsample=0.8, loss_function='Logloss',
            random_seed=0, verbose=0, thread_count=-1)

# ONE fold set shared by every config - paired_compare is invalid otherwise.
folds = CV.series_folds(sid, n_splits=5, seed=0)
step = CV.steps_from_index(index)   # derived once; run_cv would redo it per config
print('folds:', [(len(a), len(b)) for a, b in folds])

v1 43 feats | v2 48 | v3 50


folds: [(8000, 2000), (8000, 2000), (8000, 2000), (8000, 2000), (8000, 2000)]


In [3]:
def fp(kind, sl):
    """fit_predict on a column slice. Slices inside the fold so we never materialise
    a second full-size copy of X."""
    def _f(train, val):
        m = (lgb.LGBMClassifier(**LGBP) if kind == 'lgb' else CatBoostClassifier(**CATP))
        m.fit(train.X[:, sl], train.y, sample_weight=train.w)
        return m.predict_proba(val.X[:, sl])[:, 1]
    return _f

CONFIGS = [('A_lgb_v1', 'lgb', V1), ('B_cat_v1', 'cat', V1),
           ('C_cat_v2', 'cat', V2), ('D_cat_v3', 'cat', V3)]

res = {}
for name, kind, sl in CONFIGS:
    t = time.time()
    print(f'\n=== {name} ===', flush=True)
    res[name] = CV.run_cv(X, y, sid, index, fp(kind, sl), folds=folds,
                          train_row_mask=sampled, row_step=step)
    print(f'{res[name].summary(name)} | {time.time() - t:.0f}s', flush=True)


=== A_lgb_v1 ===


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  fold 0: train   263,351 rows / 8,000 series | val 1,012,996 rows / 2,000 series | TS-AUC 0.5766


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  fold 1: train   263,306 rows / 8,000 series | val   982,164 rows / 2,000 series | TS-AUC 0.5884


...

  fold 1: train   263,306 rows / 8,000 series | val   982,164 rows / 2,000 series | TS-AUC 0.5964


  fold 2: train   263,072 rows / 8,000 series | val 1,018,819 rows / 2,000 series | TS-AUC 0.5828


  fold 3: train   263,146 rows / 8,000 series | val 1,015,051 rows / 2,000 series | TS-AUC 0.6014


  fold 4: train   263,109 rows / 8,000 series | val 1,007,487 rows / 2,000 series | TS-AUC 0.6232


D_cat_v3           mean 0.5977 +/- 0.0162 (sem 0.0073) | OOF 0.5968 | folds: 0.5849  0.5964  0.5828  0.6014  0.6232 | 242s


In [4]:
print(CV.summarize(res))
print('\n--- the banked chain, link by link ---')
for a, b, claim in [('B_cat_v1', 'A_lgb_v1', 0.0059),
                    ('C_cat_v2', 'B_cat_v1', 0.0040),
                    ('D_cat_v3', 'C_cat_v2', 0.0024),
                    ('D_cat_v3', 'A_lgb_v1', 0.0123)]:
    print(f'\n[single-split claim {claim:+.4f}]')
    print(CV.paired_compare(res[a], res[b], a, b))

D_cat_v3           mean 0.5977 +/- 0.0162 (sem 0.0073) | OOF 0.5968 | folds: 0.5849  0.5964  0.5828  0.6014  0.6232
C_cat_v2           mean 0.5965 +/- 0.0163 (sem 0.0073) | OOF 0.5954 | folds: 0.5823  0.5982  0.5799  0.6021  0.6199
B_cat_v1           mean 0.5962 +/- 0.0198 (sem 0.0088) | OOF 0.5953 | folds: 0.5804  0.5934  0.5810  0.5975  0.6290
A_lgb_v1           mean 0.5929 +/- 0.0188 (sem 0.0084) | OOF 0.5926 | folds: 0.5766  0.5884  0.5787  0.5981  0.6229

--- the banked chain, link by link ---

[single-split claim +0.0059]
B_cat_v1 - A_lgb_v1: +0.0033 +/- 0.0026 (sem 0.0012, t +2.82, wins 4/5) -> likely
  per-fold deltas: +0.0038  +0.0050  +0.0023  -0.0006  +0.0061

[single-split claim +0.0040]
C_cat_v2 - B_cat_v1: +0.0002 +/- 0.0057 (sem 0.0026, t +0.09, wins 3/5) -> within noise
  per-fold deltas: +0.0019  +0.0048  -0.0011  +0.0046  -0.0091

[single-split claim +0.0024]
D_cat_v3 - C_cat_v2: +0.0013 +/- 0.0023 (sem 0.0010, t +1.21, wins 3/5) -> within noise
  per-fold deltas: +0.

In [5]:
out = {n: dict(fold_scores=r.fold_scores.tolist(), mean=r.mean, std=r.std,
               sem=r.sem, oof=r.oof_score) for n, r in res.items()}
json.dump(out, open('revalidate_results.json', 'w'), indent=2)
np.savez_compressed('oof_preds.npz', sid=sid, time=tim,
                    **{n: r.oof_pred.to_numpy() for n, r in res.items()})
print('saved revalidate_results.json + oof_preds.npz')
print(pd.DataFrame({n: r.fold_scores for n, r in res.items()}).round(4))

saved revalidate_results.json + oof_preds.npz
   A_lgb_v1  B_cat_v1  C_cat_v2  D_cat_v3
0    0.5766    0.5804    0.5823    0.5849
1    0.5884    0.5934    0.5982    0.5964
2    0.5787    0.5810    0.5799    0.5828
3    0.5981    0.5975    0.6021    0.6014
4    0.6229    0.6290    0.6199    0.6232
